# 🎵 Lab 1 — Introduction to NumPy & Pandas
## Spotify Global Music Trends Analysis

---

### 📋 Lab Overview

| Field | Details |
|---|---|
| **Lab Name** | Introduction to NumPy and Pandas |
| **Dataset** | Synthetic Spotify Global Top Tracks |
| **Domain** | Music Analytics / Data Science |
| **Tools Used** | NumPy, Pandas, Matplotlib, Seaborn |

---

### 🎯 Problem Statement

> **What audio features make a song popular on Spotify across different regions and genres?**
>
> With millions of tracks on Spotify, understanding what separates a viral hit from an average song is crucial for artists, labels, and playlist curators. In this lab, we analyze a dataset of Spotify's global top tracks to uncover patterns in audio features such as BPM, energy, danceability, loudness, and valence that correlate with streaming popularity.

### 🎓 Learning Objectives

By the end of this lab, you will be able to:
- Create and manipulate **NumPy arrays** for numerical computation
- Load, clean, and explore data using **Pandas DataFrames**
- Apply **filtering, grouping, sorting**, and aggregation on real-world style data
- Compute **descriptive statistics** and derive insights
- Visualize patterns using **Matplotlib and Seaborn**

---
## 📦 Section 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
sns.set_theme(style='whitegrid', palette='muted')

print("✅ All libraries imported successfully!")
print(f"   NumPy version  : {np.__version__}")
print(f"   Pandas version : {pd.__version__}")

---
## 🗂️ Section 2: Dataset Generation

We generate a **synthetic but realistic Spotify dataset** of 500 global top tracks with the following features:

| Feature | Description |
|---|---|
| `track_name` | Name of the song |
| `artist` | Artist name |
| `genre` | Music genre (Pop, Hip-Hop, EDM, Latin, R&B, Rock) |
| `region` | Region (US, UK, India, Brazil, Global) |
| `streams_millions` | Total streams in millions |
| `bpm` | Beats Per Minute (tempo) |
| `energy` | Energy level (0–100) |
| `danceability` | Danceability score (0–100) |
| `valence` | Musical positivity (0–100) |
| `loudness_db` | Loudness in decibels |
| `duration_sec` | Track duration in seconds |
| `explicit` | Whether the track has explicit content |
| `release_year` | Year of release |

In [ ]:
np.random.seed(42)
N = 500

genres   = ['Pop', 'Hip-Hop', 'EDM', 'Latin', 'R&B', 'Rock']
regions  = ['US', 'UK', 'India', 'Brazil', 'Global']
artists  = ['The Weeknd', 'Taylor Swift', 'Bad Bunny', 'Drake', 'Dua Lipa',
             'BTS', 'Olivia Rodrigo', 'Ed Sheeran', 'Ariana Grande', 'Billie Eilish',
             'Post Malone', 'SZA', 'Harry Styles', 'Doja Cat', 'Kendrick Lamar']

genre_arr  = np.random.choice(genres, N)
region_arr = np.random.choice(regions, N)
artist_arr = np.random.choice(artists, N)

# Audio features with genre-influenced distributions
bpm_map = {'Pop':120, 'Hip-Hop':95, 'EDM':135, 'Latin':110, 'R&B':90, 'Rock':125}
bpm = np.array([int(np.random.normal(bpm_map[g], 12)) for g in genre_arr])
bpm = np.clip(bpm, 60, 200)

energy        = np.clip(np.random.normal(65, 18, N), 0, 100).astype(int)
danceability  = np.clip(np.random.normal(70, 15, N), 0, 100).astype(int)
valence       = np.clip(np.random.normal(55, 20, N), 0, 100).astype(int)
loudness_db   = np.clip(np.random.normal(-6, 3, N), -20, 0).round(2)
duration_sec  = np.clip(np.random.normal(200, 40, N), 90, 360).astype(int)
explicit      = np.random.choice([True, False], N, p=[0.35, 0.65])
release_year  = np.random.choice(range(2018, 2025), N)

# Streams influenced by energy + danceability + some randomness
streams = (energy * 0.4 + danceability * 0.4 + np.random.normal(0, 10, N)) * np.random.uniform(0.5, 3.0, N)
streams = np.clip(streams, 1, 500).round(1)

track_names = [f"Track_{i:03d}" for i in range(1, N+1)]

df = pd.DataFrame({
    'track_name'      : track_names,
    'artist'          : artist_arr,
    'genre'           : genre_arr,
    'region'          : region_arr,
    'streams_millions': streams,
    'bpm'             : bpm,
    'energy'          : energy,
    'danceability'    : danceability,
    'valence'         : valence,
    'loudness_db'     : loudness_db,
    'duration_sec'    : duration_sec,
    'explicit'        : explicit,
    'release_year'    : release_year
})

# Inject 15 missing values randomly for cleaning exercise
for col in ['bpm', 'energy', 'danceability']:
    idx = np.random.choice(df.index, 5, replace=False)
    df.loc[idx, col] = np.nan

print(f"✅ Dataset created: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

---
## 🔢 Section 3: NumPy Fundamentals

Before working with Pandas, we explore core NumPy operations using the raw audio feature arrays extracted from the dataset.

### 3.1 — Array Creation & Properties

In [ ]:
# Extract clean arrays (drop NaNs)
energy_arr       = df['energy'].dropna().values.astype(float)
dance_arr        = df['danceability'].dropna().values.astype(float)
streams_arr      = df['streams_millions'].values
bpm_arr          = df['bpm'].dropna().values.astype(float)

print("=== NumPy Array Properties ===")
print(f"Energy array shape   : {energy_arr.shape}")
print(f"Data type            : {energy_arr.dtype}")
print(f"Number of dimensions : {energy_arr.ndim}")
print(f"Total elements       : {energy_arr.size}")

# 2D array: stacking audio features
audio_matrix = np.column_stack([energy_arr[:10], dance_arr[:10], bpm_arr[:10]])
print(f"\n2D Audio Matrix (first 10 rows × 3 features):")
print(f"Shape: {audio_matrix.shape}")
print(audio_matrix)

### 3.2 — Statistical Operations with NumPy

In [ ]:
print("=== Descriptive Statistics (NumPy) ===")
for name, arr in [('Streams (M)', streams_arr), ('Energy', energy_arr), ('Danceability', dance_arr), ('BPM', bpm_arr)]:
    print(f"\n📊 {name}")
    print(f"   Mean     : {np.mean(arr):.2f}")
    print(f"   Median   : {np.median(arr):.2f}")
    print(f"   Std Dev  : {np.std(arr):.2f}")
    print(f"   Min / Max: {np.min(arr):.2f} / {np.max(arr):.2f}")
    print(f"   25th pct : {np.percentile(arr, 25):.2f}")
    print(f"   75th pct : {np.percentile(arr, 75):.2f}")

### 3.3 — Array Operations & Broadcasting

In [ ]:
# Normalize energy and danceability to 0-1 scale using broadcasting
energy_norm = (energy_arr - energy_arr.min()) / (energy_arr.max() - energy_arr.min())
dance_norm  = (dance_arr  - dance_arr.min())  / (dance_arr.max()  - dance_arr.min())

# Composite Hype Score: weighted combination
hype_score = 0.6 * energy_norm + 0.4 * dance_norm

print("=== Hype Score (NumPy Broadcasting) ===")
print(f"Formula: 0.6 × energy_norm + 0.4 × dance_norm")
print(f"Mean Hype Score : {np.mean(hype_score):.4f}")
print(f"Top Hype Score  : {np.max(hype_score):.4f}")
print(f"Tracks above 0.8: {np.sum(hype_score > 0.8)}")

# Boolean masking
high_energy_mask = energy_arr > 80
print(f"\nHigh Energy tracks (>80): {np.sum(high_energy_mask)} / {len(energy_arr)}")

# Vectorized BPM classification
bpm_category = np.where(bpm_arr < 90, 'Slow', np.where(bpm_arr < 120, 'Medium', 'Fast'))
unique, counts = np.unique(bpm_category, return_counts=True)
print(f"\nBPM Category Distribution:")
for cat, cnt in zip(unique, counts):
    print(f"   {cat:8s}: {cnt} tracks")

### 3.4 — NumPy Linear Algebra: Correlation Matrix

In [ ]:
# Build feature matrix for correlation analysis
min_len = min(len(energy_arr), len(dance_arr), len(bpm_arr), len(streams_arr))
feature_matrix = np.vstack([
    streams_arr[:min_len],
    energy_arr[:min_len],
    dance_arr[:min_len],
    bpm_arr[:min_len]
])

corr_matrix = np.corrcoef(feature_matrix)
feature_labels = ['Streams', 'Energy', 'Danceability', 'BPM']

print("=== NumPy Correlation Matrix ===")
print(f"{'':15}", '  '.join(f"{l:12}" for l in feature_labels))
for i, label in enumerate(feature_labels):
    row = '  '.join(f"{corr_matrix[i,j]:12.4f}" for j in range(len(feature_labels)))
    print(f"{label:15} {row}")

# Visual heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', xticklabels=feature_labels,
            yticklabels=feature_labels, cmap='coolwarm', center=0)
plt.title('🎵 NumPy Correlation Matrix — Audio Features vs Streams')
plt.tight_layout()
plt.show()

---
## 🐼 Section 4: Pandas — Data Loading & Exploration

Now we use **Pandas** to perform structured data analysis on the full DataFrame.

### 4.1 — Basic Exploration

In [ ]:
print("=== Dataset Info ===")
df.info()
print("\n=== First 5 Rows ===")
display(df.head())
print("\n=== Statistical Summary ===")
display(df.describe().round(2))

### 4.2 — Missing Value Detection & Handling

In [ ]:
print("=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

# Strategy: fill with column median (robust to outliers)
for col in ['bpm', 'energy', 'danceability']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"   Filled '{col}' NaNs with median: {median_val:.1f}")

print(f"\n✅ Missing values after cleaning: {df.isnull().sum().sum()}")

### 4.3 — Feature Engineering

In [ ]:
# 1. Duration in minutes
df['duration_min'] = (df['duration_sec'] / 60).round(2)

# 2. Composite Hype Score
df['hype_score'] = (0.6 * df['energy'] + 0.4 * df['danceability']).round(2)

# 3. Popularity Tier based on streams
df['popularity_tier'] = pd.cut(
    df['streams_millions'],
    bins=[0, 50, 150, 300, 500],
    labels=['Emerging', 'Popular', 'Hit', 'Viral']
)

# 4. BPM Category
df['bpm_category'] = pd.cut(
    df['bpm'],
    bins=[0, 89, 119, 200],
    labels=['Slow', 'Medium', 'Fast']
)

print("✅ New features added:")
print(f"   → duration_min, hype_score, popularity_tier, bpm_category")
display(df[['track_name', 'genre', 'streams_millions', 'hype_score', 'popularity_tier', 'bpm_category']].head(8))

---
## 🔍 Section 5: Pandas — Filtering, Sorting & Grouping

### 5.1 — Filtering Tracks

In [ ]:
# Top 10 tracks by streams
print("=== Top 10 Most Streamed Tracks ===")
top10 = df[['track_name','artist','genre','region','streams_millions']].nlargest(10, 'streams_millions')
display(top10.reset_index(drop=True))

# Filter: High energy EDM tracks
edm_hype = df[(df['genre'] == 'EDM') & (df['energy'] > 80) & (df['bpm'] > 125)]
print(f"\n🔥 High-Energy EDM tracks (energy>80 & BPM>125): {len(edm_hype)} tracks")
display(edm_hype[['track_name','artist','bpm','energy','streams_millions']].head(5))

# Filter: Recent hits (2022 onwards, Viral tier)
recent_hits = df[(df['release_year'] >= 2022) & (df['popularity_tier'] == 'Viral')]
print(f"\n🚀 Viral tracks released in 2022+: {len(recent_hits)} tracks")

### 5.2 — GroupBy Analysis

In [ ]:
# Average audio features by Genre
print("=== Average Audio Features by Genre ===")
genre_stats = df.groupby('genre').agg(
    avg_streams = ('streams_millions', 'mean'),
    avg_energy  = ('energy', 'mean'),
    avg_dance   = ('danceability', 'mean'),
    avg_valence = ('valence', 'mean'),
    avg_bpm     = ('bpm', 'mean'),
    track_count = ('track_name', 'count')
).round(2).sort_values('avg_streams', ascending=False)
display(genre_stats)

# Streams by Region
print("\n=== Average Streams by Region ===")
region_stats = df.groupby('region')['streams_millions'].agg(['mean','median','max','count']).round(2)
display(region_stats)

# Explicit vs Non-Explicit
print("\n=== Explicit vs Non-Explicit Tracks ===")
explicit_stats = df.groupby('explicit')[['streams_millions','energy','danceability']].mean().round(2)
explicit_stats.index = ['Non-Explicit', 'Explicit']
display(explicit_stats)

### 5.3 — Pivot Tables

In [ ]:
print("=== Pivot: Average Streams by Genre × Region ===")
pivot = pd.pivot_table(
    df,
    values='streams_millions',
    index='genre',
    columns='region',
    aggfunc='mean'
).round(1)
display(pivot)

# Heatmap of pivot table
plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5)
plt.title('🌍 Average Streams (Millions) by Genre × Region')
plt.tight_layout()
plt.show()

---
## 📊 Section 6: Data Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('🎵 Spotify Music Trends — Visual Analysis', fontsize=16, fontweight='bold')

# 1. Distribution of Streams
axes[0,0].hist(df['streams_millions'], bins=30, color='steelblue', edgecolor='white')
axes[0,0].set_title('Distribution of Streams')
axes[0,0].set_xlabel('Streams (Millions)')
axes[0,0].set_ylabel('Frequency')

# 2. Average Streams by Genre
genre_stream = df.groupby('genre')['streams_millions'].mean().sort_values(ascending=False)
axes[0,1].bar(genre_stream.index, genre_stream.values, color=sns.color_palette('muted', len(genre_stream)))
axes[0,1].set_title('Avg Streams by Genre')
axes[0,1].set_ylabel('Avg Streams (M)')
axes[0,1].tick_params(axis='x', rotation=30)

# 3. Energy vs Danceability (colored by genre)
genre_colors = {'Pop':'#1DB954','Hip-Hop':'#FF5733','EDM':'#3498DB','Latin':'#F39C12','R&B':'#9B59B6','Rock':'#E74C3C'}
for genre, grp in df.groupby('genre'):
    axes[0,2].scatter(grp['energy'], grp['danceability'], alpha=0.4, s=15,
                       label=genre, color=genre_colors.get(genre, 'gray'))
axes[0,2].set_title('Energy vs Danceability by Genre')
axes[0,2].set_xlabel('Energy')
axes[0,2].set_ylabel('Danceability')
axes[0,2].legend(fontsize=7, markerscale=1.5)

# 4. Hype Score by Popularity Tier (Boxplot)
tier_order = ['Emerging', 'Popular', 'Hit', 'Viral']
df_tier = df.dropna(subset=['popularity_tier'])
tier_groups = [df_tier[df_tier['popularity_tier'] == t]['hype_score'].values for t in tier_order]
axes[1,0].boxplot(tier_groups, labels=tier_order, patch_artist=True,
                   boxprops=dict(facecolor='#1DB954', alpha=0.7))
axes[1,0].set_title('Hype Score by Popularity Tier')
axes[1,0].set_ylabel('Hype Score')

# 5. Track Count by Release Year
year_counts = df['release_year'].value_counts().sort_index()
axes[1,1].plot(year_counts.index, year_counts.values, marker='o', color='coral', linewidth=2)
axes[1,1].set_title('Tracks Released per Year')
axes[1,1].set_xlabel('Year')
axes[1,1].set_ylabel('Track Count')
axes[1,1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# 6. BPM Category Pie Chart
bpm_counts = df['bpm_category'].value_counts()
axes[1,2].pie(bpm_counts.values, labels=bpm_counts.index, autopct='%1.1f%%',
               colors=['#FF6B6B','#4ECDC4','#45B7D1'], startangle=140)
axes[1,2].set_title('BPM Category Distribution')

plt.tight_layout()
plt.show()

---
## 💡 Section 7: Key Insights & Conclusions

Based on our analysis of 500 Spotify global top tracks:

In [ ]:
print("=" * 60)
print("       🎵 SPOTIFY MUSIC TRENDS — KEY FINDINGS")
print("=" * 60)

top_genre = genre_stats['avg_streams'].idxmax()
print(f"\n🥇 Top Genre by Avg Streams  : {top_genre}")

top_region = df.groupby('region')['streams_millions'].mean().idxmax()
print(f"🌍 Top Region by Avg Streams : {top_region}")

corr_energy = df['streams_millions'].corr(df['energy'])
corr_dance  = df['streams_millions'].corr(df['danceability'])
print(f"\n📈 Correlation: Streams ↔ Energy       : {corr_energy:.4f}")
print(f"📈 Correlation: Streams ↔ Danceability : {corr_dance:.4f}")

viral_avg_hype = df[df['popularity_tier'] == 'Viral']['hype_score'].mean()
emerging_avg_hype = df[df['popularity_tier'] == 'Emerging']['hype_score'].mean()
print(f"\n🔥 Avg Hype Score — Viral tracks    : {viral_avg_hype:.2f}")
print(f"🌱 Avg Hype Score — Emerging tracks : {emerging_avg_hype:.2f}")

explicit_streams = df.groupby('explicit')['streams_millions'].mean()
print(f"\n🎤 Explicit tracks avg streams     : {explicit_streams.get(True, 0):.2f}M")
print(f"🎤 Non-explicit tracks avg streams : {explicit_streams.get(False, 0):.2f}M")

print("\n" + "=" * 60)
print("✅ Lab 1 Complete — NumPy & Pandas Analysis Done!")
print("=" * 60)

---
## 📝 Summary

In this lab, we successfully demonstrated:

1. **NumPy** — Array creation, slicing, broadcasting, statistical functions, boolean masking, and correlation matrix computation using `np.corrcoef()`.

2. **Pandas** — DataFrame creation, exploration (`info`, `describe`), missing value handling, feature engineering with `pd.cut()`, filtering with boolean conditions, groupby aggregations, and pivot tables.

3. **Visualization** — Histograms, bar charts, scatter plots, boxplots, line plots, and pie charts using Matplotlib and Seaborn.

4. **Insight** — Tracks with higher Energy and Danceability scores tend to reach the Viral popularity tier, and certain genres outperform others across specific regions — validating the value of audio feature analysis for music recommendation systems.

---
*Lab 1 | Deep Learning and Applications | Introduction to NumPy & Pandas*